<a href="https://colab.research.google.com/github/jojojojo-jo/LRP_analysis_resnet/blob/main/perturbed_tensors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#0. Import

Zennit: https://zennit.readthedocs.io/en/latest/tutorial/image-classification-vgg-resnet.html#4.3-LRP-with-custom-LayerMapComposite


In [ ]:
import torch

print(torch.__version__)

2.9.0+cu128


In [ ]:
!pip install torch==2.9.0 torchvision==0.24.0 torchaudio==2.9.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 122.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.7/124.7 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 7.2 MB/s eta 0:00:00
  Attempting uninstall: triton
    Found existing installation: triton 3.6.0
    Uninstalling triton-3.6.0:
      Successfully uninstalled triton-3.6.0
  Attempting uninstall: nvidia-nvshmem-cu12
    Found existing installation: nvidia-nvshmem-cu12 3.4.5
    Uninstalling nvidia-nvshmem-cu12-3.4.5:
      Successfully uninstalled nvidia-nvshmem-cu12-3.4.5
  Attempting uninstall: nvidia-nccl-cu12
    Found existing installation: nvidia-nccl-cu1

In [ ]:
!pip install zennit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 4.2 MB/s eta 0:00:00


In [ ]:
import torch
from torch import nn
from torch.nn import Linear
import torch.nn.functional as F

from itertools import islice

from PIL import Image

import torchvision
from torchvision import datasets, models, transforms
from torchvision.transforms import Compose, Resize, CenterCrop
from torchvision.transforms import ToTensor, Normalize
from torchvision.models import resnet18
from torchvision.models import ResNet18_Weights

from zennit.attribution import Gradient, SmoothGrad
from zennit.core import Stabilizer
from zennit.composites import EpsilonGammaBox, EpsilonPlusFlat
from zennit.composites import SpecialFirstLayerMapComposite, NameMapComposite
from zennit.image import imgify, imsave
from zennit.rules import Epsilon, ZPlus, ZBox, Norm, Pass, Flat
from zennit.types import Convolution, Activation, AvgPool, Linear as AnyLinear
from zennit.types import BatchNorm
from zennit.torchvision import ResNetCanonizer

import os
import numpy as np
import matplotlib.pyplot as plt

#import torchinfo.summary
try:
  from torchinfo import summary
except:
  print("[INFO] Couldn't find torchinfo... installing it")
  !pip install -q torchinfo
  from torchinfo import summary

#import helper functions
try:
  from going_modular.going_modular import data_setup, engine
except:
  print(f"[INFO] Couldn't find going_modular scripts... downloading from GitHub")
  !git clone https://github.com/mrdbourke/pytorch-deep-learning
  !mv pytorch-deep-learning/going_modular .
  !rm -rf pytorch-deep-learning
  from going_modular.going_modular import data_setup, engine

[INFO] Couldn't find torchinfo... installing it
[INFO] Couldn't find going_modular scripts... downloading from GitHub
Cloning into 'pytorch-deep-learning'...
remote: Enumerating objects: 4410, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 4410 (delta 5), reused 2 (delta 2), pack-reused 4400 (from 2)
Receiving objects: 100% (4410/4410), 764.18 MiB | 18.80 MiB/s, done.
Resolving deltas: 100% (2661/2661), done.
Updating files: 100% (248/248), done.


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

#1. Load trained model

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
weights = torchvision.models.ResNet18_Weights.DEFAULT
auto_transforms = weights.transforms()

model = models.resnet18(weights=weights)

num_classes = 3
model.fc = nn.Linear(model.fc.in_features, num_classes)

model_path = '/content/drive/MyDrive/study/Project/models/best_model_plateau.pth'
model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))

model.eval()

print("Model loaded successfully!")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 119MB/s]


Model loaded successfully!


#2. Generate noisy data


In [ ]:
import os
import zipfile

from pathlib import Path

import requests

data_path = Path("data/")
image_path = data_path / "pizza_steak_sushi_20_percent"

if image_path.is_dir():
  print(f"{image_path} directory exists")
else:
  print(f"Did not find {image_path} directory, creating one...")
  image_path.mkdir(parents=True, exist_ok=True)

  with open(data_path / "pizza_steak_sushi_20_percent.zip", "wb") as f:
    request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi_20_percent.zip")
    print("Downloading data...")
    f.write(request.content)

  with zipfile.ZipFile(data_path / "pizza_steak_sushi_20_percent.zip", "r") as zip_ref:
    print("Unzipping data...")
    zip_ref.extractall(image_path)

  os.remove(data_path / "pizza_steak_sushi_20_percent.zip")

Did not find data/pizza_steak_sushi_20_percent directory, creating one...
Unzipping data...


In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image
from torchvision import transforms
import torchvision.transforms.functional as TF
from tqdm import tqdm

# ==========================================
# 1. FIXED & OPTIMIZED PERTURBATION LIBRARY
# ==========================================

def get_signal_power(img_tensor):
    return torch.mean(img_tensor ** 2)

def scale_noise_by_snr(clean_img, raw_noise, target_snr_db):
    p_signal = get_signal_power(clean_img)
    p_noise_initial = torch.mean(raw_noise ** 2)
    if p_noise_initial == 0:
        return torch.zeros_like(clean_img)
    alpha = torch.sqrt(p_signal / (p_noise_initial * (10 ** (target_snr_db / 10.0))))
    return alpha * raw_noise

def apply_gaussian_noise(img_tensor, snr_db):
    raw_noise = torch.randn_like(img_tensor)
    scaled_noise = scale_noise_by_snr(img_tensor, raw_noise, snr_db)
    return torch.clamp(img_tensor + scaled_noise, 0.0, 1.0)

def apply_spectral_noise(img_tensor, snr_db, color):
    """
    Guarantees perfect spectral separation:
    - Red: Generates smooth macro color clouds via wide spatial low-pass filtering.
    - Violet: Generates jagged pixel static via edge-detection high-pass filtering.
    """
    C, H, W = img_tensor.shape
    device = img_tensor.device

    # Base source: Standard independent white noise
    raw_noise = torch.randn((C, H, W), device=device)

    if str(color).lower().strip() == 'red':
        # --- PURE LOW PASS (RED) ---
        # A large kernel size (e.g., 25) with high sigma removes ALL fine grain,
        # forcing the noise to become broad, sweeping color clouds/shadows.
        kernel_size = 25
        sigma = 5.0
        smooth_noise = TF.gaussian_blur(raw_noise, kernel_size=[kernel_size, kernel_size], sigma=[sigma, sigma])

        # Zero-mean normalization to preserve target illumination
        for c in range(C):
            smooth_noise[c] -= torch.mean(smooth_noise[c])

        scaled_noise = scale_noise_by_snr(img_tensor, smooth_noise, snr_db)

    elif str(color).lower().strip() == 'violet':
        # --- PURE HIGH PASS (VIOLET) ---
        # Subtracting a blurred version from the original acts as a perfect high-pass filter,
        # isolating nothing but rapid pixel-to-pixel transitions.
        kernel_size = 5
        sigma = 1.0
        blurred_base = TF.gaussian_blur(raw_noise, kernel_size=[kernel_size, kernel_size], sigma=[sigma, sigma])
        high_freq_noise = raw_noise - blurred_base

        for c in range(C):
            high_freq_noise[c] -= torch.mean(high_freq_noise[c])

        scaled_noise = scale_noise_by_snr(img_tensor, high_freq_noise, snr_db)

    else:
        raise ValueError(f"Unknown noise color string: {color}")

    return torch.clamp(img_tensor + scaled_noise, 0.0, 1.0)

def apply_fgsm_attack(model, img_tensor, label, epsilon):
    img_tensor = img_tensor.clone().detach().requires_grad_(True)
    output = model(img_tensor.unsqueeze(0))
    loss = F.cross_entropy(output, torch.tensor([label]).to(img_tensor.device))

    model.zero_grad()
    loss.backward()

    sign_data_grad = img_tensor.grad.data.sign()
    perturbed_image = img_tensor + epsilon * sign_data_grad
    return torch.clamp(perturbed_image, 0.0, 1.0).detach()

# ==========================================
# 2. DATA PROCESSING PIPELINE (TEST ONLY)
# ==========================================

def pipeline_process_test_dataset(base_src_dir, base_dst_dir, model=None):
    snr_levels = {
        'lvl_1_35dB': 35, 'lvl_2_25dB': 25, 'lvl_3_18dB': 18, 'lvl_4_10dB': 10, 'lvl_5_3dB': 3
    }
    fgsm_levels = {
        'lvl_1_eps009': 0.009, 'lvl_2_eps028': 0.028, 'lvl_3_eps063': 0.063, 'lvl_4_eps158': 0.158, 'lvl_5_eps354': 0.354
    }

    class_mapping = {'pizza': 0, 'steak': 1, 'sushi': 2}
    to_tensor = transforms.ToTensor()
    to_pil = transforms.ToPILImage()

    split = 'test'
    split_dir = os.path.join(base_src_dir, split)

    if not os.path.exists(split_dir):
        print(f"❌ Error: The directory '{split_dir}' does not exist.")
        return

    print("🚀 Commencing Spectral Stress-Testing Data Generation (TEST DATA ONLY)...")

    for class_name in class_mapping.keys():
        class_dir = os.path.join(split_dir, class_name)
        if not os.path.exists(class_dir):
            continue

        label = class_mapping[class_name]
        images = [f for f in os.listdir(class_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Processing Class: [{class_name}] ({len(images)} test images)")

        for img_name in tqdm(images, leave=False):
            img_path = os.path.join(class_dir, img_name)

            try:
                with Image.open(img_path).convert('RGB') as img:
                    clean_tensor = to_tensor(img)
            except Exception as e:
                print(f"Skipping corrupted file {img_name}: {e}")
                continue

            for noise_type in ['gaussian', 'red', 'violet']:
                for lvl_name, snr_val in snr_levels.items():

                    if noise_type == 'gaussian':
                        perturbed = apply_gaussian_noise(clean_tensor, snr_val)
                    else:
                        perturbed = apply_spectral_noise(clean_tensor, snr_val, color=noise_type)

                    save_dir = os.path.join(base_dst_dir, noise_type, lvl_name, split, class_name)
                    os.makedirs(save_dir, exist_ok=True)

                    perturbed_pil = to_pil(perturbed)
                    perturbed_pil.save(os.path.join(save_dir, img_name))

            if model is not None:
                device = next(model.parameters()).device
                for lvl_name, eps_val in fgsm_levels.items():
                    perturbed = apply_fgsm_attack(model, clean_tensor.to(device), label, eps_val)

                    save_dir = os.path.join(base_dst_dir, 'fgsm', lvl_name, split, class_name)
                    os.makedirs(save_dir, exist_ok=True)

                    perturbed_pil = to_pil(perturbed.cpu())
                    perturbed_pil.save(os.path.join(save_dir, img_name))

    print("✅ Test data perturbations successfully generated with corrected spectral mappings.")

# ==========================================
# 3. EXECUTION
# ==========================================
SOURCE_DIR = "data/pizza_steak_sushi_20_percent"
DESTINATION_DIR = "data_perturbed"

pipeline_process_test_dataset(SOURCE_DIR, DESTINATION_DIR, model=model)

🚀 Commencing Spectral Stress-Testing Data Generation (TEST DATA ONLY)...
Processing Class: [pizza] (46 test images)


Processing Class: [steak] (58 test images)


Processing Class: [sushi] (46 test images)


✅ Test data perturbations successfully generated with corrected spectral mappings.


#3. Generate LRP Heatmaps

In [ ]:
# define the base image transform
transform_img = Compose([
    Resize(256),
    CenterCrop(224),
])
# define the normalization transform
transform_norm = Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
# define the full tensor transform
transform = Compose([
    transform_img,
    ToTensor(),
    transform_norm,
])

In [ ]:
from pathlib import Path
import re
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import pandas as pd

# --- Configuration ---
perturbed_base_dir = Path('data_perturbed')
output_tensor_dir = Path('output_tensors')   # NEW: raw relevance .npy files

class_to_idx = {'pizza': 0, 'steak': 1, 'sushi': 2}
idx_to_class = {v: k for k, v in class_to_idx.items()}

# Folder naming conventions observed:
#   fgsm:     lvl_<N>_eps<DDD>   e.g. lvl_1_eps009 -> level 1, eps digits "009"
#   gaussian/red/violet: lvl_<N>_<DD>dB e.g. lvl_1_35dB -> level 1, 35 dB
#
# EPS_SCALE controls how the zero-padded eps digits map to a real epsilon
# value. As generated (eps009, eps028, eps063, eps158, eps354), dividing by
# 1000 gives an increasing epsilon sequence (0.009 -> 0.354), which is the
# standard assumption for pixel range [0,1]. Adjust if your generator used a
# different convention.
EPS_SCALE = 1000.0

LEVEL_RE = re.compile(r'lvl_(\d+)_(.+)')
EPS_RE = re.compile(r'eps0*(\d+)')
DB_RE = re.compile(r'(\d+)\s*dB', re.IGNORECASE)


def parse_intensity_level(noise_type: str, lvl_name: str) -> dict:
    """
    Parses a level folder name into structured numeric fields.

    Returns dict with:
        level_number   -> int, the 'N' in lvl_N_... (ordinal, 1=weakest by convention)
        intensity_value -> float, numeric epsilon or SNR(dB)
        intensity_unit  -> 'epsilon' | 'dB' | 'unknown'
    """
    level_match = LEVEL_RE.match(lvl_name)
    level_number = int(level_match.group(1)) if level_match else None
    remainder = level_match.group(2) if level_match else lvl_name

    eps_match = EPS_RE.search(remainder)
    db_match = DB_RE.search(remainder)

    if eps_match is not None:
        raw_digits = eps_match.group(1)
        intensity_value = float(raw_digits) / EPS_SCALE
        intensity_unit = 'epsilon'
    elif db_match is not None:
        intensity_value = float(db_match.group(1))
        intensity_unit = 'dB'
    else:
        intensity_value = np.nan
        intensity_unit = 'unknown'

    return {
        'level_number': level_number,
        'intensity_value': intensity_value,
        'intensity_unit': intensity_unit,
    }


model.to(device)
model.eval()

canonizer = ResNetCanonizer()
composite = EpsilonPlusFlat(canonizers=[canonizer])

inference_records = []

with Gradient(model=model, composite=composite) as attributor:

    for noise_path in perturbed_base_dir.iterdir():
        if not noise_path.is_dir():
            continue
        noise_type = noise_path.name

        for lvl_path in noise_path.iterdir():
            if not lvl_path.is_dir():
                continue
            lvl_name = lvl_path.name
            intensity_info = parse_intensity_level(noise_type, lvl_name)

            split_dir = lvl_path / 'test'
            if not split_dir.exists():
                continue

            for folder_class, class_idx in class_to_idx.items():
                source_dir = split_dir / folder_class
                if not source_dir.exists():
                    continue

                img_paths = [p for p in source_dir.glob('*') if p.suffix.lower() in ['.jpg', '.jpeg', '.png']]

                print(f"Generating LRP for: {noise_type} | {lvl_name} | Class: {folder_class} ({len(img_paths)} images)")

                for img_path in img_paths:
                    try:
                        image = Image.open(img_path).convert('RGB')
                        data = transform(image)[None].to(device)

                        with torch.no_grad():
                            output = model(data)
                            probabilities = torch.softmax(output, dim=1)
                            max_prob, pred_idx = torch.max(probabilities, dim=1)
                            pred_int = pred_idx.item()
                            confidence_score = max_prob.item()
                            pred_class_name = idx_to_class.get(pred_int, "unknown")

                        target_tensor = torch.eye(len(class_to_idx), device=device)[[pred_int]]

                        output, attribution = attributor(data, target_tensor)

                        # Raw relevance, shape (1, H, W) -> (H, W)
                        relevance = attribution.sum(1).cpu()
                        relevance_np = relevance.squeeze(0).numpy().astype(np.float32)

                        # --- NEW: Save raw relevance tensor (.npy) ---
                        tensor_target_dir = output_tensor_dir / noise_type / lvl_name / 'test' / folder_class
                        tensor_target_dir.mkdir(parents=True, exist_ok=True)
                        tensor_save_path = tensor_target_dir / f"tensor_{img_path.stem}.npy"
                        np.save(tensor_save_path, relevance_np)

                        inference_records.append({
                            'image_name': img_path.name,
                            'image_stem': img_path.stem,
                            'noise_type': noise_type,
                            'intensity_level': lvl_name,
                            'level_number': intensity_info['level_number'],
                            'intensity_value': intensity_info['intensity_value'],
                            'intensity_unit': intensity_info['intensity_unit'],
                            'ground_truth_class': folder_class,
                            'predicted_class': pred_class_name,
                            'confidence_score': confidence_score,
                            'tensor_path': str(tensor_save_path),
                        })

                    except Exception as e:
                        print(f"Error processing {img_path.name} under {noise_type}/{lvl_name}: {e}")

df_metrics = pd.DataFrame(inference_records)
print(f"✅ Process Complete!  Tensors -> {output_tensor_dir}")

Generating LRP for: gaussian | lvl_2_25dB | Class: pizza (46 images)


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:841: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:270.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Generating LRP for: gaussian | lvl_2_25dB | Class: steak (58 images)
Generating LRP for: gaussian | lvl_2_25dB | Class: sushi (46 images)
Generating LRP for: gaussian | lvl_4_10dB | Class: pizza (46 images)
Generating LRP for: gaussian | lvl_4_10dB | Class: steak (58 images)
Generating LRP for: gaussian | lvl_4_10dB | Class: sushi (46 images)
Generating LRP for: gaussian | lvl_5_3dB | Class: pizza (46 images)
Generating LRP for: gaussian | lvl_5_3dB | Class: steak (58 images)
Generating LRP for: gaussian | lvl_5_3dB | Class: sushi (46 images)
Generating LRP for: gaussian | lvl_1_35dB | Class: pizza (46 images)
Generating LRP for: gaussian | lvl_1_35dB | Class: steak (58 images)
Generating LRP for: gaussian | lvl_1_35dB | Class: sushi (46 images)
Generating LRP for: gaussian | lvl_3_18dB | Class: pizza (46 images)
Generating LRP for: gaussian | lvl_3_18dB | Class: steak (58 images)
Generating LRP for: gaussian | lvl_3_18dB | Class: sushi (46 images)
Generating LRP for: red | lvl_2_25dB 

#6. Save data & output

In [ ]:
import shutil

source_path = '/content/output_tensors'
destination_path = '/content/drive/MyDrive/study/Project/LRP_Heatmap/perturbed_tensors'
shutil.copytree(source_path, destination_path, dirs_exist_ok=True)

print("Output saved to drive")

Output saved to drive
